# Rough prototype: traceable PDF RAG

This notebook records the experiments performed before the production application. The goal is to answer five questions: Can Docling recover text and provenance? What chunk size looks reasonable? Are local embeddings useful? Does reranking change the order? Can a hosted model answer using only labeled evidence?

The notebook deliberately uses a small page range and NumPy cosine search. The application later replaces the experimental array with persistent Qdrant HNSW search.

In [ ]:
from pathlib import Path
import os, time, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
load_dotenv(ROOT / '.env')
DATA = ROOT / 'data' / 'corpus'
MODELS = ROOT / 'models'
DATA.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
SEED = 7
np.random.seed(SEED)
print('Project:', ROOT)

## 1. Acquire one representative report

The full application validates all ten reports. For rapid experimentation this notebook downloads one official IPCC report and converts only the first 20 pages.

In [ ]:
PDF_URL = 'https://www.ipcc.ch/site/assets/uploads/sites/2/2022/06/SR15_Full_Report_HR.pdf'
PDF_PATH = DATA / 'IPCC_SR15_Full_Report.pdf'
if not PDF_PATH.exists():
    partial = PDF_PATH.with_suffix('.pdf.part')
    headers = {'User-Agent': 'TraceablePdfRag/0.1 (educational document retrieval demo)'}
    with requests.get(PDF_URL, headers=headers, stream=True, timeout=(15, 180)) as response:
        response.raise_for_status()
        with partial.open('wb') as handle:
            for block in response.iter_content(1024 * 1024):
                if block:
                    handle.write(block)
    assert partial.read_bytes()[:5] == b'%PDF-'
    partial.replace(PDF_PATH)
print(PDF_PATH, f'{PDF_PATH.stat().st_size / 1024**2:.1f} MB')

In [ ]:
from pypdf import PdfReader
reader = PdfReader(str(PDF_PATH))
native_lengths = [len((reader.pages[i].extract_text() or '').strip()) for i in range(20)]
print('Total pages:', len(reader.pages))
pd.Series(native_lengths, name='native characters').describe()

## 2. Layout-aware extraction and provenance

Docling is the document pipeline; Tesseract is the OCR engine it calls for image regions. PDF-aware OCR preserves good native text instead of replacing every page with noisier OCR.

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TesseractOcrOptions, HeadingHierarchyOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption

options = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    generate_parsed_pages=True,
    ocr_options=TesseractOcrOptions(lang=['eng']),
)
options.table_structure_options.mode = TableFormerMode.FAST
options.heading_hierarchy_options = HeadingHierarchyOptions(enabled=True)
converter = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)})
started = time.perf_counter()
document = converter.convert(PDF_PATH, page_range=(1, 20)).document
print(f'Converted 20 pages in {time.perf_counter() - started:.1f}s')

In [ ]:
rows = []
for item, _level in document.iterate_items():
    text = getattr(item, 'text', '')
    prov = getattr(item, 'prov', []) or []
    rows.append({
        'type': type(item).__name__,
        'page': prov[0].page_no if prov else None,
        'bbox': tuple(round(v, 1) for v in (prov[0].bbox.l, prov[0].bbox.t, prov[0].bbox.r, prov[0].bbox.b)) if prov else None,
        'text': text[:120],
    })
items = pd.DataFrame(rows)
display(items.head(15))
display(items['type'].value_counts().to_frame('count'))

## 3. Chunk-size experiment

The hybrid chunker preserves document sections and only splits where the token limit requires it. Comparing several sizes exposes the recall/context tradeoff instead of choosing a number without evidence.

In [ ]:
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

MODEL_NAME = 'BAAI/bge-small-en-v1.5'
experiments = {}
for limit in (400, 700, 1000):
    tokenizer = HuggingFaceTokenizer.from_pretrained(model_name=MODEL_NAME, max_tokens=limit)
    chunker = HybridChunker(tokenizer=tokenizer, merge_peers=True)
    chunks = list(chunker.chunk(document))
    records = []
    for chunk in chunks:
        text = chunker.contextualize(chunk)
        pages = sorted({p.page_no for item in chunk.meta.doc_items for p in item.prov})
        records.append({'text': text, 'tokens': tokenizer.count_tokens(text), 'pages': pages, 'headings': chunk.meta.headings})
    experiments[limit] = records
summary = pd.DataFrame([{'limit': limit, 'chunks': len(records), 'median_tokens': np.median([r['tokens'] for r in records]), 'max_tokens': max(r['tokens'] for r in records)} for limit, records in experiments.items()])
display(summary)
for limit, records in experiments.items():
    plt.hist([r['tokens'] for r in records], bins=15, alpha=.45, label=str(limit))
plt.xlabel('tokens per chunk'); plt.ylabel('count'); plt.legend(title='limit'); plt.show()

**Prototype decision:** start production at 700 tokens with 105 tokens (15%) overlap. This is large enough to preserve explanatory context but keeps five retrieved passages within a compact generation prompt. The final value must still be tuned with R@5 and MRR.

## 4. Local embeddings and simple cosine baseline

This is intentionally an in-memory exact baseline. The application uses Qdrant HNSW for persistence, metadata filters and scaling.

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(MODEL_NAME, cache_folder=str(MODELS / 'huggingface'))
prototype_chunks = experiments[700]
texts = [row['text'] for row in prototype_chunks]
started = time.perf_counter()
matrix = embedder.encode_document(texts, normalize_embeddings=True, convert_to_numpy=True)
print(matrix.shape, f'embedded in {time.perf_counter() - started:.2f}s')

In [ ]:
QUESTION = 'What impacts are projected when global warming reaches 1.5 degrees Celsius?'
query = embedder.encode_query(QUESTION, normalize_embeddings=True, convert_to_numpy=True)
scores = matrix @ query
top_indices = np.argsort(scores)[::-1][:12]
dense_results = [{**prototype_chunks[i], 'index': int(i), 'dense_score': float(scores[i])} for i in top_indices]
display(pd.DataFrame([{'rank': n+1, 'score': r['dense_score'], 'pages': r['pages'], 'text': r['text'][:180]} for n, r in enumerate(dense_results)]))

## 5. Reranking experiment

Dense retrieval compares separately created vectors. A cross-encoder jointly reads the question and each of the twelve candidates, trading a small amount of CPU time for better final ordering.

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
started = time.perf_counter()
rerank_scores = reranker.predict([(QUESTION, row['text']) for row in dense_results])
for row, score in zip(dense_results, rerank_scores):
    row['rerank_score'] = float(score)
reranked = sorted(dense_results, key=lambda row: row['rerank_score'], reverse=True)[:5]
print(f'Reranked 12 passages in {(time.perf_counter() - started)*1000:.0f}ms')
display(pd.DataFrame([{'rank': n+1, 'dense': r['dense_score'], 'reranker': r['rerank_score'], 'pages': r['pages'], 'text': r['text'][:180]} for n, r in enumerate(reranked)]))

## 6. Grounded answer with application-owned citations

The model receives opaque labels. Page numbers are retained by code and are never accepted from generated text. The cell skips cleanly when no Groq key is configured.

In [ ]:
source_blocks = []
source_map = {}
for number, row in enumerate(reranked, start=1):
    label = f'S{number}'
    source_map[label] = {'filename': PDF_PATH.name, 'pages': row['pages']}
    source_blocks.append(f'[{label}] pages {row["pages"]}\n{row["text"]}')
context = '\n\n'.join(source_blocks)
api_key = os.getenv('GROQ_API_KEY')
if not api_key:
    print('Skipped generation: add GROQ_API_KEY to .env')
else:
    from groq import Groq
    response = Groq(api_key=api_key).chat.completions.create(
        model=os.getenv('GROQ_MODEL', 'openai/gpt-oss-20b'),
        temperature=0, max_tokens=220,
        messages=[
            {'role':'system','content':'Answer only from the passages. Cite every factual paragraph with supplied [S#] labels. If unsupported, say: Insufficient evidence in the indexed reports.'},
            {'role':'user','content':f'Context:\n{context}\n\nQuestion: {QUESTION}'},
        ],
    )
    answer = response.choices[0].message.content
    print(answer)
    display(source_map)

## 7. Prototype findings → application design

- Layout-aware conversion provides the page and bounding-box provenance required for traceable citations.
- The 700-token experiment is a reasonable initial balance, with 15% overlap added in production for boundary continuity.
- Exact NumPy cosine search is useful for validation but has no durable metadata store or ANN index, so production uses Qdrant HNSW.
- Reranking is restricted to twelve candidates because applying a cross-encoder to the full corpus would violate latency goals.
- Source labels must resolve through application metadata. Allowing the model to invent filenames or pages would make citations untrustworthy.
- Expensive extraction and embedding belong in an offline cached pipeline; only query embedding, retrieval, reranking and generation belong in the 2–5 second path.
- The hosted generator is acceptable for this public corpus. A private production corpus needs an approved private or local model.